# 1Point3Acres Web Scraper

We would like to access more colloquial code switching data. To do so, let's access the web forum 1Point3Acres. It's a popular forum for English-Mandarin bilingual speakers to discuss career and academic related topics.

# Requests

In [ ]:
!pip install requests
!pip install beautifulsoup4

import requests
from bs4 import BeautifulSoup
import time
import re
import csv
import pandas as pd

Let's navigate to the 1point3acres homepage. The BeautifulSoup library lets us get the HTML as a text file that we can parse through.

In [ ]:
oneptthree_home_page = requests.get('https://www.1point3acres.com/bbs/')
soup = BeautifulSoup(oneptthree_home_page.text, 'html.parser')

topic_spans = soup.find_all('span', class_='titletext')[4:22]
topics = [t.text for t in topic_spans]
forum_links = [t.find('a', href=True)["href"] for t in topic_spans]
print(len(forum_links))
print(forum_links)

Now, we have all of the links to the subforums and the topic names stored. Next, let's go to each of the individual forum pages and get the threads located there. We want to also ignore certain words that 1point3acres inserts into posts, as these were not typed by the poster.

In [ ]:
ign = ['x',
       '1point3acres',
       'com',
       '1point',
       '3acres',
       'baidu',
       'bbs',
       'waral',
       'www']
ignored_words = set(ign)

Let's define a few helper methods for our cleaning process.

In [ ]:
def find_english_words(phrase):
  if not phrase:
    return 0, ''
  string_split_regex = u'[\u4e00-\u9fff]|[a-zA-Z0-9]+'
  arr = re.findall(string_split_regex, phrase)
  ar = [i for i in arr if i]

  temp_eng_words = []
  for word in arr:
    chars = re.search('[a-zA-Z]', str(word))
    if chars!=None and word not in temp_eng_words and word not in ignored_words:
      temp_eng_words.append(word)

  if len(temp_eng_words) == 0:
    return 0, ''
  else:
    return 1, temp_eng_words

Now, let's go through each topic page and get the posts. This is gonna take a really really long time to run (about 20-30 minutes).

In [ ]:
thread_topics, titles, dates, eng_words_title, eng_words_thread, blended_title, blended_post, links, post_bodies = [], [], [], [], [], [], [], [], []
num_blended = 0

'''
First, we loop through all of the forums
'''

for index, forum in enumerate(forum_links):
  print(forum)
  forum_page = requests.get('https://www.1point3acres.com/bbs/' + forum)
  time.sleep(1)
  p = BeautifulSoup(forum_page.text, 'html.parser')
  threads = p.find_all('tbody')

  '''
  Some of the forum pages are structured differently.
  For convenience, we only look at the ones that are all structured the same.
  Perhaps down the road, can come back and code for differently structured forum pages.
  '''
  if len(threads) < 5:
    continue

  '''
  For each of the threads on the page, we get the title, date, and blended words
  '''
  for t in threads:
    title = t.find('a', href=True, class_='s xst')
    if not title:
      continue

    '''
    Find all the English words in the title
    '''
    # print('title: ', title.text)
    blend, temp_eng_words = find_english_words(title.text)
    blended_title.append(blend)
    eng_words_title.append(temp_eng_words)
    # print('title eng words: ', temp_eng_words)
    num_blended = num_blended + 1 if blend==1 else num_blended

    '''
    We store the thread link for later, in order to get the post messsage
    '''
    thread_link = t.find('a', href=True, class_='s xst')['href']
    # print('thread link: ', 'https://www.1point3acres.com/bbs/' + thread_link)
    links.append('https://www.1point3acres.com/bbs/' + thread_link)

    '''
    Find the date that the thread was initially published
    '''
    td = t.find('td', class_='by').find('span')
    if td.find('span'):
      td = td.find('span')['title']
    else:
      td = td.text

    thread_topics.append(topics[index])
    titles.append(title.text)
    dates.append(td)

    thread_page = requests.get('https://www.1point3acres.com/bbs/' + thread_link)
    time.sleep(5)
    thread_post_parser = BeautifulSoup(thread_page.text, 'html.parser')
    post_body = thread_post_parser.find('td', class_='t_f')
    if post_body:
      post_body = post_body.text
    post_bodies.append(post_body)
    blend, temp_eng_words = find_english_words(post_body)
    blended_post.append(blend)
    eng_words_thread.append(temp_eng_words)
    # print('body eng words: ', temp_eng_words, '\n')
    num_blended = num_blended + 1 if blend==1 else num_blended



Using all the data we have collected, let's make our initial CSV

In [ ]:
csv = {'Topic': thread_topics,
        'Thread Date': dates,
        'Thread Title': titles,
        'Title Blended': blended_title,
        'English Words Title': eng_words_title,
        'Thread Body': post_bodies,
        'Thread Blended': blended_post,
        'English Words Thread': eng_words_thread,
        'Thread Link': links
        }

for k in csv:
  print(len(csv[k]))
csv_df = pd.DataFrame(csv)
csv_df.to_csv('1point3acres_threads.csv')
print('number of blended phrases: ', num_blended)

Now that we have our file, let's try to assess how many Chinese words there are. There are a couple of different ways we can try to do this.

# Dictionary based segmentation

First, let's use a [list](https://raw.githubusercontent.com/lwinmoe/segment/master/chinese-word-list.txt) of words in Chinese to determine how many words there are (credit to [this](https://github.com/lwinmoe/segment) Github repository). Make sure that both this file and the CSV generated from the previous step are uploaded in the files section on the left.

In [ ]:
chinese_word_list = set(line.strip() for line in open('chinese-word-list.txt'))
print('Number of words in dictionary: ', len(chinese_word_list))

test_phrase = '地里之前也有帖子聊过这个问题，但是时间点停留在19年了，保险起见再确认一下'

index=0
test_phrase_word_list = []

while index < len(test_phrase)-1:
  curr_word = test_phrase[index:index+2]
  # print(curr_word)
  if curr_word in chinese_word_list:
    test_phrase_word_list.append(curr_word)
    index+=2
  else:
    test_phrase_word_list.append(curr_word[0])
    index+=1

print(test_phrase_word_list)

While this may work for this particular sentence, it's not entirely accurate as words in Chinese are not always made of 2 characters. Let's try out a different software that has been built to solve this problem.

# SpaCy POS tagging and word segmentation

[POS tagging](https://en.wikipedia.org/wiki/Part-of-speech_tagging) is a computational technique designed to identify what part of speech words in a sentence are.

We want to tag both the English words and Chinese words in the sentence.
Let's try the [spaCy](https://melaniewalsh.github.io/Intro-Cultural-Analytics/05-Text-Analysis/Multilingual/Chinese/03-POS-Keywords-Chinese.html) POS tagger for the Chinese words. This will not only segment the words for us, but will also give us additional data.

In [ ]:
!pip install -U spacy
import spacy
from spacy import displacy
from collections import Counter
pd.set_option("max_rows", 400)
pd.set_option("max_colwidth", 400)
!python -m spacy download zh_core_web_md
pos = spacy.load('zh_core_web_md')

In [ ]:
test_phrase = '哪里可以用最快速度排到Rolex？'
doc = pos(test_phrase)
print('test phrase: ', test_phrase, '\n')

for token in doc:
    print(token, token.pos_)

Now, let's do this for all our entries

In [ ]:
csvs = [pd.read_csv('1point3acres_segmented_threads_dec9.csv', index_col=[0]),
        pd.read_csv('1point3acres_segmented_threads_nov29.csv', index_col=[0])]

for f in csvs:
  words_pos = []
  for index, thr in f.iterrows():
    thread_pos = []
    thread_title = thr['Sentence']
    thread_title_pos = pos(thread_title)
    for token in thread_title_pos:
      thread_pos.append((token.text, token.pos_))
    words_pos.append(thread_pos)
  f['Sentence Parts of Speech'] = words_pos

csvs[0].to_csv('1point3acres_threads_dec9_pos.csv')
csvs[1].to_csv('1point3acres_threads_nov29_pos.csv')

Now, let's finally put together our final CSV. We will combine the data from December 9 and November 29, and include the following columns:

- Thread ID
- Thread Topic
- Sentence
- Sentence English Words
- Number English Words
- Sentence Chinese Words
- Number Chinese Words
- POS Tagging
- Proportion of English Words
- Code-switching

Let's make sure the two files from the last step are uploaded. Check that 1point3acres_threads_dec9_pos and 1point3acres_threads_nov29_pos are uploaded on the lefthand side

In [ ]:
nov_29 = pd.read_csv('1point3acres_threads_nov29_pos.csv', index_col=[0])
dec_9 = pd.read_csv('1point3acres_threads_dec9_pos.csv', index_col=[0])

num_threads_nov29 = nov_29["Thread ID"]
num_threads_nov29 = num_threads_nov29[len(num_threads_nov29)-1]

thread_id = nov_29["Thread ID"].append(dec_9["Thread ID"]+num_threads_nov29, ignore_index=True)
thread_topic = nov_29["Thread Topic"].append(dec_9["Thread Topic"], ignore_index=True)
sentence = nov_29["Sentence"].append(dec_9["Sentence"], ignore_index=True)
eng_words = nov_29["Sentence English Words"].append(dec_9["Sentence English Words"], ignore_index=True)
num_eng_words = nov_29["Number English Words"].append(dec_9["Number English Words"], ignore_index=True)
ch_words = nov_29["Sentence Chinese Words"].append(dec_9["Sentence Chinese Words"], ignore_index=True)
num_ch_words = nov_29["Number Chinese Words"].append(dec_9["Number Chinese Words"], ignore_index=True)
num_ch_chars = nov_29["Number Chinese Characters"].append(dec_9["Number Chinese Characters"], ignore_index=True)
pos_tagging = nov_29["Sentence Parts of Speech"].append(dec_9["Sentence Parts of Speech"], ignore_index=True)
thread_link = nov_29["Thread Link"].append(dec_9["Thread Link"], ignore_index=True)

code_switching = []
prop_code_switching = []
count_code_switching = 0

for index, e_word_length in enumerate(num_eng_words):
  if e_word_length != 0 and num_ch_words[index] != 0:
    count_code_switching+=1
    code_switching.append(1)
  else:
    code_switching.append(0)
  total_words = e_word_length+num_ch_words[index]
  prop_code_switching.append(e_word_length/total_words)


print(count_code_switching)

csv = {'thread_id': thread_id,
        'thread_topic': thread_topic,
        'sentence': sentence,
        'eng_words': eng_words,
        'num_eng_words': num_eng_words,
        'ch_words': ch_words,
        'num_ch_words': num_ch_words,
        'num_ch_chars': num_ch_chars,
        'pos_tagging': pos_tagging,
        'cs': code_switching,
        'proportion_cs': prop_code_switching,
        'thread_link': thread_link,
        }
csv_df = pd.DataFrame(csv)
csv_df.to_csv('corpus.csv')

# Analysis

Now, let's do some analytical work.

There's a number of ways to calculate the code-switching proportion. This helper method will calculate the proportion of code-switching in a given topic (remember that each phrase in our corpus is labelled with its topic).

In [ ]:
def cs_prop_topic(csv, topic):
  num_eng_words, num_ch_words, num_total_words, avg_cs_prop, total_sentences = 0,0,0,0,0
  num_ch_sentences, num_eng_sentences, num_cs_sentences = 0,0,0
  num_ch_cs_words, num_eng_cs_words,num_cs_words = 0,0,0
  for index, sentence in csv.iterrows():
    if sentence['thread_topic'] != topic:
      continue
    total_sentences += 1
    eng, ch = sentence['num_eng_words'], sentence['num_ch_words']
    num_eng_words += eng
    num_ch_words += ch
    tw = ch+eng
    num_total_words += tw
    avg_cs_prop += sentence['proportion_cs']
    if sentence['proportion_cs'] == 0: #only in chinese
      num_ch_sentences += 1
    elif sentence['proportion_cs'] == 1:
      num_eng_sentences += 1
    else:
      num_cs_sentences += 1
      num_eng_cs_words += eng
      num_ch_cs_words += ch
      num_cs_words += tw
  avg_cs_prop = avg_cs_prop/total_sentences
  return num_eng_words, num_ch_words, num_total_words, avg_cs_prop, total_sentences, num_ch_sentences, num_eng_sentences, num_cs_sentences, num_eng_cs_words, num_ch_cs_words, num_cs_words


Here's a summary of our topic-based results. We calculate the number of English and Mandarin words based off of the segmentation results from spaCy.

In [ ]:
import pandas as pd
import regex as re
import ast

cor pus = pd.read_csv('corpus.csv', index_col=[0], converters={'ch_words': ast.literal_eval, 'pos_tagging': ast.literal_eval})

for index, sentence in corpus.iterrows():
  #for each sentence, find num eng and ch words from spacy and replace in column
  pos = sentence['pos_tagging']
  eng, ch = [], []
  num_eng, num_ch = 0,0
  for w in pos:
    word = w[0]
    word_pos = w[1]
    if word_pos == 'PUNCT' or word_pos == 'SYM':
      continue
    elif re.search('[a-zA-Z]', word):
      eng.append(word)
      num_eng += 1
    elif re.findall(r'[\u4e00-\u9fff]+', word):
      ch.append(word)
      num_ch += 1

  corpus.at[index, 'eng_words'] = eng
  corpus.at[index, 'num_eng_words'] = num_eng
  corpus.at[index, 'ch_words'] = ch
  corpus.at[index, 'num_ch_words'] = num_ch

corpus.to_csv('corpus_spacy_replace.csv')


In [ ]:
import pandas as pd
import regex as re
import ast

corpus = pd.read_csv('corpus.csv', index_col=[0], converters={'ch_words': ast.literal_eval, 'pos_tagging': ast.literal_eval})

for index, sentence in corpus.iterrows():
  eng = sentence['num_eng_words']
  total = eng + sentence['num_ch_words']
  corpus.at[index, 'proportion_cs'] = eng/total

corpus.to_csv('corpus_cs.csv')


Now, let's count the amount of code-switching relative to each part of speech. We do this by first, for each topic, finding all English words, then counting up the number of parts of speech. We then divide the counts of parts of speech by the number of total English words in each topic. This gives us the code-switching proportion of each POS per topic. Here's a helper method to help with that:

In [ ]:
def num_pos_per_topic(csv, topic):
  noun, verb, adj, adv, adp, aux, conj, cconj, det, intj, num, part, pron, propn, sconj = 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
  count_eng = 0
  for index, sentence in csv.iterrows():
    if sentence['thread_topic'] != topic:
      continue
    if sentence['cs'] == 0:
      continue
    pos_tagging = sentence['pos_tagging']
    for tag in pos_tagging:
      word = tag[0]
      pos = tag[1]
      if re.search('[a-zA-Z]', word) is None: #we only want to look at english words
        continue
      if word == 'From':
        continue
      count_eng += 1
      if pos == 'ADJ':
        adj += 1
      elif pos == 'ADP':
        adp += 1
      elif pos == 'ADV':
        adv += 1
      elif pos == 'AUX':
        aux += 1
      elif pos == 'CONJ':
        conj += 1
      elif pos == 'CCONJ':
        cconj += 1
      elif pos == 'DET':
        det += 1
      elif pos == 'INTJ':
        intj += 1
      elif pos == 'NOUN':
        adp += 1
      elif pos == 'NUM':
        num += 1
      elif pos == 'PART':
        part += 1
      elif pos == 'PRON':
        pron += 1
      elif pos == 'PROPN':
        propn += 1
      elif pos == 'SCONJ':
        sconj += 1

  return noun/count_eng, verb/count_eng, adj/count_eng, adv/count_eng, adp/count_eng, aux/count_eng, conj/count_eng, cconj/count_eng, det/count_eng, intj/count_eng, num/count_eng, part/count_eng, pron/count_eng, propn/count_eng, sconj/count_eng

And now, here's a summary of the results:

In [ ]:
import pandas as pd
import regex as re
import ast

corpus = pd.read_csv('corpus.csv', index_col=[0], converters={'ch_words': ast.literal_eval, 'pos_tagging': ast.literal_eval})

print('Summary of Topic Based Results: \n')
list_topics = ['EECS', '国内职位内推', '好物Deals', '数据科学', '留学申请', '留美生活', '签证手续', '职位内推', '职场达人', '院系介绍']
for t in list_topics:
  noun, verb, adj, adv, adp, aux, conj, cconj, det, intj, num, part, pron, propn, sconj = num_pos_per_topic(corpus, t)
  print('Topic: ', t)
  print('Nouns: ', noun)
  print('Verbs: ', verb)
  print('Adjectives: ', adj)
  print('Adverbs: ', adv)
  print('Adpositions: ', adp)
  print('Auxiliary: ', aux)
  print('Conjunction: ', conj)
  print('Coordinating Conjunction: ', cconj)
  print('Determiner: ', det)
  print('Interjection: ', intj)
  print('Numeral: ', num)
  print('Particle: ', part)
  print('Pronoun: ', pron)
  print('Proper noun: ', propn)
  print('Suboordinating conjunction: ', sconj)
  print('-----------------------------------------')
  list_pos = [('Noun: ', noun),
              ('Verb: ', verb),
              ('Adjective: ', adj),
              ('Adverb: ', adv),
              ('Adposition: ', adp),
              ('Auxiliary: ', aux),
              ('Conjunction: ', conj),
              ('Coordinating Conjunction: ', cconj),
              ('Determiner: ', det),
              ('Interjection: ', intj),
              ('Numeral: ', num),
              ('Particle: ', part),
              ('Pronoun: ', pron),
              ('Proper noun: ', propn),
              ('Suboordinating conjunction: ', sconj)]
  list_pos.sort(reverse = True, key = lambda x: x[1])
  print('Top 3 parts of speech: ')
  print(list_pos[0][0], list_pos[0][1])
  print(list_pos[1][0], list_pos[1][1])
  print(list_pos[2][0], list_pos[2][1])
  print('\n')
